[Reference](https://medium.com/@Rohan_Dutt/10-genai-techniques-for-orchestrating-multi-stage-etl-pipelines-6ab28814c6c5$0)

# 1. Dynamic Task Routing with AI Decision Trees

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
import joblib
from typing import Dict, Any

class DataRouter:
    def __init__(self, model_path: str = "router.pkl"):
        self.model = joblib.load(model_path)
        self.pipelines = {
            "clean": self.run_light_pipeline,
            "messy": self.run_heavy_pipeline,
            "structured": self.run_validation_pipeline
        }

    def extract_features(self, data: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame({
            "null_pct": [data.isnull().sum().sum() / data.size],
            "unique_ratio": [len(data.columns) / max(len(data), 1)],
            "json_nested_depth": [self._max_json_depth(data.to_dict())]
        })

    def _max_json_depth(self, obj, depth=0):
        if not isinstance(obj, dict): return depth
        return max([self._max_json_depth(v, depth+1) for v in obj.values()], default=depth)

    def route_and_process(self, data: pd.DataFrame) -> Dict[str, Any]:
        features = self.extract_features(data)
        route = self.model.predict(features)[0]
        return {"route": route, "result": self.pipelines[route](data)}

    def run_light_pipeline(self, df): return df.dropna().to_dict()
    def run_heavy_pipeline(self, df): return df.fillna(0).astype(str).to_dict()
    def run_validation_pipeline(self, df): return df.describe().to_dict()

# Training workflow (run once)
def train_router(historical_data, labels):
    router = RandomForestClassifier(n_estimators=50)
    features = pd.concat([DataRouter().extract_features(d) for d in historical_data])
    router.fit(features, labels)
    joblib.dump(router, "router.pkl")

# Production usage
router = DataRouter()
incoming_data = pd.read_json("data.json")
result = router.route_and_process(incoming_data)  # Auto-routed in 50ms

# 2. AI-Powered Anomaly Triaging

In [2]:
import pandas as pd
from langchain.agents import Tool, AgentExecutor
from langchain.chat_models import ChatOpenAI
from langchain.tools import BaseTool

class AnomalyTriageAgent:
    def __init__(self, data_source: str):
        self.llm = ChatOpenAI(temperature=0)
        self.data = pd.read_parquet(data_source)
        self.tools = [
            Tool(name="check_api_status", func=self._check_api,
                 description="Checks external API uptime"),
            Tool(name="query_data_volume", func=self._query_volume,
                 description="Gets record counts for time periods"),
            Tool(name="detect_schema_drift", func=self._detect_schema,
                 description="Compares current vs historical schema")
        ]
        self.agent = AgentExecutor.from_agent_and_tools(
            agent="zero-shot-react-description",
            tools=self.tools,
            llm=self.llm,
            verbose=True
        )

    def _check_api(self, timestamp: str):
        # Simulate API status check
        return "Shopify API experienced 15min outage at 2024-01-15 14:30 UTC"

    def _query_volume(self, time_range: str):
        subset = self.data[self.data['timestamp'].str.contains(time_range)]
        return f"Volume: {len(subset)} records (normal: 500-600)"

    def _detect_schema(self, _):
        return "New field 'discount_code' detected in 12% of records"

    def triage(self, alert: dict) -> str:
        prompt = f"""
        Anomaly detected: {alert}
        Investigate root cause using available tools.
        Provide 2-sentence explanation with confidence score.
        """
        return self.agent.run(prompt)

# Production integration
agent = AnomalyTriageAgent("production_logs.parquet")

alert = {
    "metric": "transaction_volume",
    "spike": 300,
    "timestamp": "2024-01-15 14:35"
}

explanation = agent.triage(alert)
# Output: "Root cause: Shopify API outage caused delayed processing
# and subsequent backlog flush. Confidence: 95%"

# 3. Self-Healing Schema Mapping


In [3]:
from fuzzywuzzy import fuzz
import json
from collections import defaultdict

class SchemaHealer:
    def __init__(self, schema_history: dict):
        self.canonical_map = self._build_canonical_map(schema_history)

    def _build_canonical_map(self, history: dict) -> dict:
        map = defaultdict(list)
        for canonical, variants in history.items():
            for variant in variants:
                map[variant].append(canonical)
        return map

    def heal_schema(self, incoming_data: dict, threshold: int = 85) -> dict:
        healed = {}
        for key, value in incoming_data.items():
            if key in self.canonical_map:
                healed[self.canonical_map[key][0]] = value
            else:
                matched = self._fuzzy_match(key)
                if matched:
                    healed[matched] = value
                    self._learn_mapping(matched, key)
                else:
                    healed[key] = value
        return healed

    def _fuzzy_match(self, key: str) -> str:
        best_match, best_score = None, 0
        for variant, canonicals in self.canonical_map.items():
            score = fuzz.ratio(key.lower(), variant.lower())
            if score > 85 and score > best_score:
                best_match = canonicals[0]
                best_score = score
        return best_match

    def _learn_mapping(self, canonical: str, variant: str):
        if variant not in self.canonical_map:
            self.canonical_map[variant].append(canonical)

# Usage
schema_history = {
    "customer_id": ["cust_id", "customerid", "c_id"],
    "order_total": ["ord_ttl", "total", "order_amount"]
}
healer = SchemaHealer(schema_history)

# Messy incoming data
raw_data = {"cust_id": 12345, "ord_ttl": 99.99, "random_field": "x"}
cleaned = healer.heal_schema(raw_data)
# Output: {"customer_id": 12345, "order_total": 99.99, "random_field": "x"}

# 4. Synthetic Data for Pipeline Stress Testing


In [4]:
import pandas as pd
from datetime import datetime
import numpy as np
from faker import Faker
from langchain.chat_models import ChatOpenAI
from langchain.prompts import PromptTemplate

class SyntheticStressTester:
    def __init__(self, base_schema: dict):
        self.fake = Faker()
        self.base_schema = base_schema
        self.llm = ChatOpenAI(temperature=0.7)

    def generate_edge_cases(self, n_cases: int = 100) -> pd.DataFrame:
        cases = []
        for _ in range(n_cases):
            case = {}
            for field, ftype in self.base_schema.items():
                case[field] = self._generate_stress_value(ftype)
            cases.append(case)
        return pd.DataFrame(cases)

    def _generate_stress_value(self, ftype: str):
        if ftype == "email":
            return np.random.choice([self.fake.email(), "invalid@", None, ""])
        elif ftype == "timestamp":
            return np.random.choice([
                datetime.now().isoformat(),
                "0000-00-00 00:00:00",  # Invalid date
                None,
                "2021-02-30 25:70:90"   # Malformed
            ])
        elif ftype == "amount":
            return np.random.choice([999999999.99, -50.00, None, 0, 0.001])
        return self.fake.word()

    def generate_llm_edge_cases(self, domain: str, prompt: str) -> list:
        template = PromptTemplate(
            input_variables=["domain", "count"],
            template="Generate {count} plausible but problematic {domain} records with edge cases"
        )
        return self.llm.predict(template.format(domain=domain, count=10))

# Usage
tester = SyntheticStressTester({
    "email": "email",
    "transaction_time": "timestamp",
    "amount": "amount"
})

# Generate 1000 edge cases
stress_df = tester.generate_edge_cases(1000)
stress_df.to_parquet("stress_test_data.parquet", compression="gzip")

# Test your pipeline
def test_pipeline_resilience():
    df = pd.read_parquet("stress_test_data.parquet")
    try:
        result = your_etl_pipeline(df)
        assert len(result) > 0
    except Exception as e:
        print(f"Pipeline broke on: {e}")  # Fix this before prod

# 5. “Just-in-Time” Data Enrichment


In [5]:
import requests
from functools import lru_cache
import hashlib
import time

class JITEnricher:
    def __init__(self, cache_ttl: int = 3600):
        self.cache = {}
        self.cache_ttl = cache_ttl
        self.api_calls = 0

    @lru_cache(maxsize=1024)
    def get_currency_rate(self, from_cur: str, to_cur: str) -> float:
        # LRU cache layer
        cache_key = f"{from_cur}_{to_cur}"
        if cache_key in self.cache:
            rate, timestamp = self.cache[cache_key]
            if time.time() - timestamp < self.cache_ttl:
                return rate

        self.api_calls += 1
        response = requests.get(
            f"https://api.exchangerate.host/latest?base={from_cur}&symbols={to_cur}",
            timeout=5
        )
        rate = response.json()["rates"][to_cur]

        self.cache[cache_key] = (rate, time.time())
        return rate

    def enrich_transaction(self, tx: dict) -> dict:
        # Only enrich international transactions
        if tx["currency"] == "USD":
            return tx

        # JIT enrichment
        rate = self.get_currency_rate(tx["currency"], "USD")
        tx["usd_amount"] = tx["amount"] * rate
        tx["enrichment_timestamp"] = time.time()
        return tx

    def bulk_enrich(self, transactions: list) -> list:
        return [self.enrich_transaction(tx) for tx in transactions]

# Usage
enricher = JITEnricher(cache_ttl=1800)  # 30min TTL

transactions = [
    {"id": 1, "amount": 100, "currency": "USD"},
    {"id": 2, "amount": 95, "currency": "EUR"},
    {"id": 3, "amount": 150, "currency": "USD"}
]

enriched = enricher.bulk_enrich(transactions)
print(f"API calls made: {enricher.api_calls}")  # Only 1 call for EUR

# 6. Autonomous Retry Logic with AI


In [6]:
import time
from collections import deque
import boto3
from typing import Callable

class AIRetryEngine:
    def __init__(self):
        self.failure_memory = deque(maxlen=1000)  # Rolling window
        self.cloudwatch = boto3.client("cloudwatch")

    def log_failure(self, error_type: str, region: str, wait_time: int, success: bool):
        self.failure_memory.append({
            "error": error_type,
            "region": region,
            "wait_time": wait_time,
            "success": success,
            "timestamp": time.time()
        })

    def get_optimal_retry_strategy(self, error: str, region: str) -> dict:
        relevant = [f for f in self.failure_memory if f["error"] == error]
        if not relevant:
            return {"wait": 30, "action": "retry_same_region"}

        # Simple RL-inspired logic
        successes = [f for f in relevant if f["success"]]
        if len(successes) / len(relevant) > 0.7:
            avg_wait = sum(f["wait_time"] for f in successes) / len(successes)
            return {"wait": avg_wait, "action": "retry_same_region"}
        else:
            return {"wait": 120, "action": "failover_to_secondary_region"}

    def execute_with_retry(self, func: Callable, *args, kwargs):
        max_attempts = 3
        for attempt in range(max_attempts):
            try:
                result = func(*args, kwargs)
                if attempt > 0:
                    self.log_failure("transient", "us-east-1", 2attempt, True)
                return result
            except Exception as e:
                strategy = self.get_optimal_retry_strategy(type(e).__name__, "us-east-1")
                time.sleep(strategy["wait"])
                if attempt == max_attempts - 1:
                    self.log_failure(type(e).__name__, "us-east-1", strategy["wait"], False)
                    raise

# Usage
engine = AIRetryEngine()

def flaky_api_call():
    import random
    if random.random() < 0.6:
        raise TimeoutError("API timeout")
    return "success"

# Auto-retries with dynamic wait
result = engine.execute_with_retry(flaky_api_call)

# 7. Embedding-Driven Data Prioritization

In [7]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

class PriorityOrchestrator:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model = SentenceTransformer(model_name)
        self.priority_embeddings = self._define_priority_vectors()

    def _define_priority_vectors(self):
        return {
            "high": self.model.encode("VIP customer high transaction value regulatory critical"),
            "medium": self.model.encode("standard customer normal transaction"),
            "low": self.model.encode("test data internal low value")
        }

    def score_batch(self, batch_df: pd.DataFrame) -> pd.DataFrame:
        # Create metadata string for each record
        batch_df["metadata_text"] = (
            batch_df["customer_tier"] + " " +
            batch_df["transaction_type"] + " " +
            batch_df["regulatory_flag"].astype(str)
        )

        embeddings = self.model.encode(batch_df["metadata_text"].tolist())

        scores = []
        for emb in embeddings:
            similarities = {
                level: cosine_similarity([emb], [level_emb])[0][0]
                for level, level_emb in self.priority_embeddings.items()
            }
            scores.append(max(similarities, key=similarities.get))

        batch_df["priority"] = scores
        batch_df["priority_score"] = np.select(
            [batch_df["priority"] == "high", batch_df["priority"] == "medium"],
            [3, 2], 1
        )
        return batch_df

    def process_with_qos(self, data: pd.DataFrame) -> dict:
        scored = self.score_batch(data)
        high_priority = scored[scored["priority_score"] == 3]
        normal_priority = scored[scored["priority_score"] < 3]

        # Process in priority order
        results = {
            "critical": self._fast_track_pipeline(high_priority),
            "standard": self._standard_pipeline(normal_priority)
        }
        return results

    def _fast_track_pipeline(self, df): return len(df)  # Simulate processing
    def _standard_pipeline(self, df): return len(df)

# Usage
orchestrator = PriorityOrchestrator()
transactions = pd.DataFrame({
    "customer_tier": ["VIP", "Standard", "VIP", "Test"],
    "transaction_type": ["wire", "ach", "wire", "internal"],
    "regulatory_flag": [True, False, True, False]
})

results = orchestrator.process_with_qos(transactions)
# VIP wires processed first, test data queued last

# 8. GenAI-Generated Pipeline Documentation


In [8]:
import ast
from langchain.chat_models import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
import json

class DocumentationGenerator:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4-turbo-preview")
        self.prompt = ChatPromptTemplate.from_template(
            """Analyze this Python ETL code and generate:
            1. One-line description
            2. Input/output schemas
            3. Critical dependencies
            4. Failure modes

            Code: {code}

            Return JSON with keys: description, inputs, outputs, dependencies, failure_modes"""
        )

    def parse_pipeline_code(self, file_path: str) -> dict:
        with open(file_path, "r") as f:
            source = f.read()

        tree = ast.parse(source)
        functions = [node.name for node in ast.walk(tree) if isinstance(node, ast.FunctionDef)]
        imports = [node.names[0].name for node in ast.walk(tree) if isinstance(node, ast.Import)]

        return {"functions": functions, "imports": imports, "source": source}

    def generate_docs(self, file_path: str) -> dict:
        parsed = self.parse_pipeline_code(file_path)

        chain = self.prompt | self.llm
        response = chain.invoke({"code": parsed["source"]})

        docs = json.loads(response.content)
        docs["functions"] = parsed["functions"]
        docs["last_generated"] = json.dumps(ast.literal_eval(response.json()))

        # Write to markdown
        with open("PIPELINE_DOCS.md", "w") as f:
            f.write(f"# {file_path} Documentation\n")
            f.write(f"{docs['description']}\n")
            f.write(f"Critical Failure: {docs['failure_modes'][0]}\n")

        return docs

# Usage
generator = DocumentationGenerator()
docs = generator.generate_docs("etl_pipeline.py")

# Auto-post to Slack
import slack_sdk
slack_client = slack_sdk.WebClient(token="xoxb-token")
slack_client.chat_postMessage(
    channel="#data-engineering",
    text=f"Pipeline docs updated: {docs['description']}"
)

# 9. “Dark Launch” Parallel Pipelines

In [9]:
import pandas as pd
import hashlib
from typing import Dict, Any

class DarkLaunchManager:
    def __init__(self, experiment_id: str):
        self.experiment_id = experiment_id
        self.results_store = {}

    def run_parallel_pipelines(self, data: pd.DataFrame) -> Dict[str, Any]:
        # Production pipeline (control)
        control_result = self._production_pipeline(data.copy())

        # Shadow pipeline (treatment) with AI-suggested optimization
        treatment_result = self._optimized_pipeline(data.copy())

        # Compare results with tolerance
        comparison = self._compare_results(control_result, treatment_result)

        # Log everything
        self._log_experiment({
            "experiment_id": self.experiment_id,
            "control": control_result["metrics"],
            "treatment": treatment_result["metrics"],
            "match": comparison["match"],
            "efficiency_gain": comparison["efficiency_delta"]
        })

        return {"production": control_result, "shadow": treatment_result}

    def _production_pipeline(self, df: pd.DataFrame) -> dict:
        start = time.time()
        result = df.groupby("customer_id").agg({"amount": "sum"}).to_dict()
        return {"metrics": {"duration": time.time() - start, "rows": len(df)}}

    def _optimized_pipeline(self, df: pd.DataFrame) -> dict:
        start = time.time()
        # AI suggestion: Use smaller data types and vectorized operations
        df["amount"] = df["amount"].astype("float32")
        result = df.groupby("customer_id")["amount"].sum().to_dict()
        return {"metrics": {"duration": time.time() - start, "rows": len(df)}}

    def _compare_results(self, control: dict, treatment: dict) -> dict:
        # Hash comparison with tolerance
        control_hash = hashlib.md5(str(control).encode()).hexdigest()
        treatment_hash = hashlib.md5(str(treatment).encode()).hexdigest()

        return {
            "match": control_hash == treatment_hash,
            "efficiency_delta": (control["metrics"]["duration"] - treatment["metrics"]["duration"]) / control["metrics"]["duration"]
        }

    def _log_experiment(self, data: dict):
        pd.DataFrame([data]).to_parquet(
            f"experiments/{self.experiment_id}_{int(time.time())}.parquet"
        )

# Usage
manager = DarkLaunchManager("partition_optimization")

# Process 10% of traffic through shadow pipeline
if hashlib.md5(str(time.time()).encode()).hexdigest()[-1] in "012":
    results = manager.run_parallel_pipelines(df)
else:
    results = manager._production_pipeline(df)  # Normal path

# 10. AI-Optimized Cost Guardrails


In [10]:
import pandas as pd
from sklearn.linear_model import Ridge
import boto3
from typing import List

class CostGuardrail:
    def __init__(self, cost_threshold: float = 50.0):
        self.model = Ridge(alpha=1.0)
        self.cost_threshold = cost_threshold
        self.cloudwatch = boto3.client("cloudwatch")
        self._train_model()

    def _train_model(self):
        # Load historical pipeline runs
        history = pd.read_parquet("pipeline_runs_history.parquet")

        features = history[[
            "input_rows", "input_mb", "spark_executors",
            "shuffle_partitions", "complexity_score"
        ]]
        labels = history["cost_usd"]

        self.model.fit(features, labels)
        self.feature_names = features.columns.tolist()

    def predict_cost(self, job_config: dict) -> float:
        features = pd.DataFrame([{
            "input_rows": job_config["rows"],
            "input_mb": job_config["size_mb"],
            "spark_executors": job_config["executors"],
            "shuffle_partitions": job_config["partitions"],
            "complexity_score": job_config["complexity"]
        }])

        predicted = self.model.predict(features)[0]
        confidence = 1.0  # Simplified

        return predicted, confidence

    def enforce_guardrail(self, job_config: dict) -> dict:
        predicted_cost, _ = self.predict_cost(job_config)

        if predicted_cost > self.cost_threshold * 1.5:
            return {"action": "block", "reason": f"Predicted cost ${predicted_cost:.2f} exceeds 1.5x threshold"}
        elif predicted_cost > self.cost_threshold:
            return {"action": "alert", "reason": f"Cost warning: ${predicted_cost:.2f}"}

        # Set autoscaling limits
        self._apply_cost_optimization(job_config)
        return {"action": "allow", "optimized_config": job_config}

    def _apply_cost_optimization(self, config: dict):
        # AI suggestion: Reduce executors if input small
        if config["size_mb"] < 1000 and config["executors"] > 5:
            config["executors"] = 3

        self.cloudwatch.put_metric_alarm(
            AlarmName=f"PipelineCost_{config['job_id']}",
            MetricName="EstimatedCost",
            Threshold=self.cost_threshold,
            ActionsEnabled=True,
            AlarmActions=["arn:aws:sns:cost-alerts"]
        )

# Usage
guardrail = CostGuardrail(cost_threshold=75.00)

job_config = {
    "job_id": "daily_etl_001",
    "rows": 5000000,
    "size_mb": 2500,
    "executors": 10,
    "partitions": 200,
    "complexity": 7.5
}

decision = guardrail.enforce_guardrail(job_config)
# Output: {"action": "alert", "reason": "Cost warning: $89.50"}